# Phase 1: OR breakout chronology and one-change stop/target test

This notebook extends the 30-minute OR study. The 1-, 2-, or 5-minute **completed close** beyond the OR line is the signal; fill at the next available one-minute candle **open**. Use the first breakout of either direction per ticker/day/timeframe, chosen before outcomes. Exclude November 4–5 as in earlier studies. No QQQ, ticker-gap, or time-of-week filters are added in this pass.

**A. Descriptive path:** Measure maximum continuation **strictly before** the first touch back to the OR line, time to touch, and depth through the line afterward, from one-minute bars *after* signal close. The touch minute is excluded from the pre-return extension because its internal order is unknown. If the first bar touches the line, the pre-return extension is 0. No return means post-return penetration is missing. Report 15/30/60-minute and session-end windows separately; missing data audit is saved.

**B. Trade replay:** One hypothetical position per ticker/day/timeframe, filled at the next minute's open. Baseline 1% stop, 2% target; 5 basis points of adverse slippage on entry and exit, stop-first for bars touching both stop and target. Hard exit at 15:55 ET (last observed one-minute close before 15:55). A target hit is a win; stop hit or time exit at −0.5% or worse is a loss; other timed exits neutral. R is net return divided by chosen stop fraction. We separately vary **target** with the 1% stop fixed, and **stop** with the 2% target fixed. Each comparison uses the exact same preselected signal events and fill prices. Report W/L/N and net R for each cell, upper/lower separately. Borrow availability and actual spreads are not modeled.

**Interpretation:** Descriptive continuation does not tell us whether a 2% target comes *before* a 1% stop; the trade replay does. One-minute OHLC lacks within-minute ordering: stop-first is conservative. A target reached in the entry minute is ignored unless it is reached again in a later minute, because we cannot tell whether it preceded the open fill; the stop remains active in the entry minute. Before adopting a change, compare by month/date and reserve later weeks for validation. Historical backtests are hypothetical.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd, numpy as np, re
from IPython.display import display
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive')
ROOT=next((p for p in (BASE/'Backtesting',BASE/'backtesting') if p.is_dir()),None)
if ROOT is None: raise FileNotFoundError('Backtesting folder missing in MyDrive')
OUT=BASE/'OR_Phase1_Chronology_Stop_Target'; OUT.mkdir(exist_ok=True)
TZ='America/New_York'
OR_MIN=30
SIGNAL_LAST='15:00'
SESSION_END='15:55'
HORIZONS=(15,30,60,'session_end')
TIMEFRAMES=(1,2,5)
EXCLUDED={'2026-11-04','2026-11-05'}
paths={}
csv_candidates=list(ROOT.rglob('*_1m_*.csv'))
print('Data folder:',ROOT)
print('Candidate one-minute files:',len(csv_candidates))
print('Example paths:',[str(f.relative_to(ROOT)) for f in csv_candidates[:3]])
for f in csv_candidates:
    m=re.match(r'(.+)_1m_[0-9]{4}-[0-9]{2}-[0-9]{2}$',f.stem,re.I)
    if m: paths.setdefault(m.group(1).upper(),[]).append(f)
print('Tickers:',len(paths),'one-minute CSV files:',sum(map(len,paths.values())))
if not paths:
    raise FileNotFoundError('No matching CSVs. Check Data folder and Example paths above; reopen the latest notebook if running a saved Colab copy.')


In [ ]:
def sessions_for_ticker(file_list):
    parts={}
    for f in sorted(file_list):
        try:
            x=pd.read_csv(f)
            tc=next((c for c in x if c.lower() in ('datetime','timestamp','date')),x.columns[0])
            t=pd.to_datetime(x[tc],utc=True,errors='coerce').dt.tz_convert(TZ)
            y=pd.DataFrame({'time':t})
            for c in ('Open','High','Low','Close','Volume'):
                y[c]=pd.to_numeric(x[c],errors='coerce') if c in x else np.nan
            y=y.dropna(subset=['time','Open','High','Low','Close'])
            y=y.set_index('time').between_time('09:30','15:59')
            for date,g in y.groupby(y.index.strftime('%Y-%m-%d')):
                parts.setdefault(date,[]).append(g)
        except Exception as e: print('Skipped',f.name,str(e)[:90])
    out={}
    for date,groups in parts.items():
        d=pd.concat(groups).sort_index()
        d=d[~d.index.duplicated(keep='last')]
        out[date]=d
    return out

def event_candidates(d,date,minutes,high,low):
    start=pd.Timestamp(date+' 09:30',tz=TZ)
    if minutes==1: bars=d[['Open','High','Low','Close']].copy()
    else:
        bars=d[['Open','High','Low','Close']].resample(
            f'{minutes}min',origin=start,label='left',closed='left'
        ).agg({'Open':'first','High':'max','Low':'min','Close':'last'})
        count=d.Close.resample(f'{minutes}min',origin=start,label='left',closed='left').count()
        bars=bars[count.eq(minutes)].dropna()
    prior={'upper':False,'lower':False}
    events=[]
    for stamp,c in bars.iterrows():
        signal=stamp+pd.Timedelta(minutes=minutes)
        if signal<start+pd.Timedelta(minutes=OR_MIN): continue
        if signal>pd.Timestamp(date+' '+SIGNAL_LAST,tz=TZ):break
        upper=bool(c.Close>high);lower=bool(c.Close<low)
        if upper and not prior['upper']:
            events.append(dict(signal_time=signal,side='upper',boundary=high,signal_close=float(c.Close)))
        if lower and not prior['lower']:
            events.append(dict(signal_time=signal,side='lower',boundary=low,signal_close=float(c.Close)))
        prior={'upper':upper,'lower':lower}
    return events



In [ ]:
SLIPPAGE_BPS=5
STOP_GRID=(.0075,.01,.0125)
TARGET_GRID=(.01,.015,.02,.025,.03)
PLANS=[('baseline',.01,.02)]
PLANS += [('target_only',.01,t) for t in TARGET_GRID if t!=.02]
PLANS += [('stop_only',s,.02) for s in STOP_GRID if s!=.01]

def directional_path(d,date,e,width,horizon):
    signal=e['signal_time']
    cutoff=pd.Timestamp(date+' '+SESSION_END,tz=TZ)
    stop_at=min(cutoff,signal+pd.Timedelta(minutes=horizon)) if isinstance(horizon,int) else cutoff
    expected=pd.date_range(signal,stop_at,freq='min',inclusive='left')
    if not len(expected):return None
    x=d.reindex(expected)
    if x[['Open','High','Low','Close']].isna().any().any():return None
    sign=1 if e['side']=='upper' else -1
    boundary=e['boundary']
    outward=(x.High-boundary) if sign==1 else (boundary-x.Low)
    inward=(x.Low-boundary) if sign==1 else (boundary-x.High)
    touched=inward.le(0)
    if touched.any():
        k=int(np.flatnonzero(touched.to_numpy())[0])
        pre=x.iloc[:k]
        pre_out=outward.iloc[:k]
        first=expected[k]
        depth=max(0.,float(-inward.iloc[k:].min()))
        extension=max(0.,float(pre_out.max())) if len(pre) else 0.
        post_out=max(0.,float(outward.iloc[k:].max()))
    else:
        first=pd.NaT;depth=np.nan;extension=max(0.,float(outward.max()));post_out=np.nan
    inside=((x.Close-boundary)*sign).lt(0)
    def pct(v):return 100*v/boundary if np.isfinite(v) else np.nan
    return dict(horizon=str(horizon),return_touch=bool(touched.any()),
       return_close_inside=bool(inside.any()),return_time=first,
       minutes_to_touch=(first-signal).total_seconds()/60 if pd.notna(first) else np.nan,
       extension_before_return_dollars=extension,
       extension_before_return_pct=pct(extension),
       extension_before_return_or_width=extension/width,
       penetration_after_return_dollars=depth,
       penetration_after_return_pct=pct(depth),
       penetration_after_return_or_width=depth/width if np.isfinite(depth) else np.nan,
       extension_after_return_dollars=post_out)

def trade_replay(d,date,e,stop_frac,target_frac):
    signal=e['signal_time']
    end=pd.Timestamp(date+' '+SESSION_END,tz=TZ)
    expected=pd.date_range(signal,end,freq='min',inclusive='left')
    if not len(expected):return None,'late_signal'
    x=d.reindex(expected)
    if x[['Open','High','Low','Close']].isna().any().any():
        return None,'incomplete_trade_path'
    sign=1 if e['side']=='upper' else -1
    # No fill on missing open; complete path required for all hypothetical plans.
    raw=float(x.iloc[0].Open)
    entry=raw*(1+sign*SLIPPAGE_BPS/10000)
    stop=entry*(1-sign*stop_frac)
    target=entry*(1+sign*target_frac)
    outcome='time';exit_raw=float(x.iloc[-1].Close);exit_time=x.index[-1]
    for j,(t,c) in enumerate(x.iterrows()):
        hit_stop=bool(c.Low<=stop) if sign==1 else bool(c.High>=stop)
        hit_target=bool(c.High>=target) if sign==1 else bool(c.Low<=target)
        if hit_stop:
            outcome='stop';exit_raw=stop;exit_time=t;break
        if hit_target and j>0:
            outcome='target';exit_raw=target;exit_time=t;break
    exit_price=exit_raw*(1-sign*SLIPPAGE_BPS/10000)
    net_frac=sign*(exit_price-entry)/entry
    result='win' if outcome=='target' else 'loss' if outcome=='stop' or net_frac<=-.005 else 'neutral'
    return dict(entry_time=signal,entry=entry,exit_time=exit_time,exit=exit_price,
                outcome=outcome,result=result,net_pct=100*net_frac,R=net_frac/stop_frac),None

In [ ]:
path_rows=[];trade_rows=[];audit=[]
for count,ticker in enumerate(sorted(paths),1):
    for date,d in sessions_for_ticker(paths[ticker]).items():
        if date in EXCLUDED: continue
        start=pd.Timestamp(date+' 09:30',tz=TZ)
        opening=d.reindex(pd.date_range(start,periods=OR_MIN,freq='min'))
        if opening[['Open','High','Low','Close']].isna().any().any():
            audit.append(dict(ticker=ticker,date=date,reason='incomplete_OR'));continue
        hi=float(opening.High.max());lo=float(opening.Low.min());width=hi-lo
        if width<=0:continue
        for tf in TIMEFRAMES:
            events=event_candidates(d,date,tf,hi,lo)
            if not events:continue
            e=events[0] # first signal, either direction, before looking at results
            base=dict(ticker=ticker,date=date,timeframe_min=tf,side=e['side'],
                      signal_time=e['signal_time'],boundary=e['boundary'],
                      signal_close=e['signal_close'],or_high=hi,or_low=lo,or_width=width)
            for horizon in HORIZONS:
                row=directional_path(d,date,e,width,horizon)
                if row is not None:path_rows.append({**base,**row})
                else:audit.append(dict(ticker=ticker,date=date,timeframe_min=tf,
                                       horizon=str(horizon),reason='incomplete_path'))
            for family,stop,target in PLANS:
                row,reason=trade_replay(d,date,e,stop,target)
                if row is not None:trade_rows.append({**base,'family':family,
                    'stop_pct':100*stop,'target_pct':100*target,**row})
                else:audit.append(dict(ticker=ticker,date=date,timeframe_min=tf,
                    family=family,reason=reason))
    if count%100==0:print('Loaded',count,'/',len(paths),'tickers')
paths_df=pd.DataFrame(path_rows);trades=pd.DataFrame(trade_rows);coverage=pd.DataFrame(audit)
if paths_df.empty or trades.empty: raise RuntimeError('No complete events; inspect data root and audit')
paths_df.to_csv(OUT/'phase1_chronological_paths.csv',index=False)
trades.to_csv(OUT/'phase1_trade_replay.csv',index=False)
coverage.to_csv(OUT/'phase1_coverage_audit.csv',index=False)
print('Path-window rows',len(paths_df),'trade-plan rows',len(trades),
      'unique ticker-days',trades[['ticker','date']].drop_duplicates().shape[0])

In [ ]:
def path_summary(g):
    ret=g[g.return_touch]
    return pd.Series(dict(n=len(g),tickers=g.ticker.nunique(),dates=g.date.nunique(),
        returns=int(g.return_touch.sum()),return_pct=round(100*g.return_touch.mean(),1),
        closes_inside=int(g.return_close_inside.sum()),
        med_pre_return_extension_pct=g.extension_before_return_pct.median(),
        p75_pre_return_extension_pct=g.extension_before_return_pct.quantile(.75),
        med_minutes_to_return=ret.minutes_to_touch.median(),
        med_post_return_penetration_pct=ret.penetration_after_return_pct.median(),
        p75_post_return_penetration_pct=ret.penetration_after_return_pct.quantile(.75)))
p=paths_df.groupby(['timeframe_min','side','horizon'],dropna=False).apply(
    path_summary,include_groups=False).reset_index()
p.to_csv(OUT/'phase1_path_summary.csv',index=False)
display(p.round(3))
def trade_summary(g):
    return pd.Series(dict(trades=len(g),wins=int(g.result.eq('win').sum()),
        losses=int(g.result.eq('loss').sum()),neutrals=int(g.result.eq('neutral').sum()),
        win_pct=round(100*g.result.eq('win').mean(),1),
        net_R=round(g.R.sum(),2),mean_R=round(g.R.mean(),3),
        median_net_pct=round(g.net_pct.median(),3),
        tickers=g.ticker.nunique(),dates=g.date.nunique()))
summary=trades.groupby(['timeframe_min','side','family','stop_pct','target_pct'],
                       dropna=False).apply(trade_summary,include_groups=False).reset_index()
assert (summary.wins+summary.losses+summary.neutrals==summary.trades).all()
summary.to_csv(OUT/'phase1_trade_summary.csv',index=False)
display(summary)
monthly=trades.assign(month=trades.date.str[:7]).groupby(
    ['month','timeframe_min','side','family','stop_pct','target_pct'],
    dropna=False).apply(trade_summary,include_groups=False).reset_index()
monthly.to_csv(OUT/'phase1_trade_summary_by_month.csv',index=False)
dates=trades.groupby(['date','timeframe_min','side','family','stop_pct','target_pct'],
    dropna=False).apply(trade_summary,include_groups=False).reset_index()
dates.to_csv(OUT/'phase1_trade_summary_by_date.csv',index=False)
print('Outputs:',OUT)

## How to decide whether to change the baseline

Compare target-only plans to 1% stop / 2% target within a timeframe and direction; compare stop-only plans to the same baseline. All plans draw from the identical preselected event set. Check **win/loss/neutral**, total and average R, fills, ticker/date coverage, and month-by-month stability. A larger target can raise R while reducing win rate; a wider stop changes the R denominator. Do not combine a winning target change and a winning stop change until each has been assessed separately and a later data period validates the combination.

This first pass is a next-candle-open breakout strategy. Your OR-line limit strategy needs its own separate replay because entry and unfilled orders differ. Do not transfer the winning target/stop automatically to that strategy.